# A4-M2 실습 — 쿼리 문법을 Python으로

- 이 노트북 옆에 `.env`(OPENSEARCH_PASSWORD)를 둡니다. `.env.example`을 복사해 만드세요.
- Dev Tools에서 완성한 쿼리를 `body=`에 그대로 넣어 실행합니다. 교안 10탭, 13탭 미션 5.

In [1]:
%pip install opensearch-py python-dotenv pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 셀 1 · 로그인 방식으로 연결 (A4-M1 13탭)

In [7]:
import os
from pathlib import Path
from dotenv import load_dotenv
from opensearchpy import OpenSearch

env_path = Path.cwd() / ".env"

print("현재 실행 폴더 :", Path.cwd())
print(".env 존재 여부 :", env_path.exists())

load_dotenv(env_path, override=True)

password = os.getenv("OPENSEARCH_PASSWORD")

print("비밀번호 로드 :", "성공" if password else "실패")

client = OpenSearch(
    hosts=[{"host": "localhost", "port": 9200}],
    http_auth=("admin", password),
    use_ssl=True,
    verify_certs=False,
    ssl_show_warn=False,
)

print("sales 문서 수 :", client.count(index="sales")["count"])

현재 실행 폴더 : d:\OneDrive\#classes\November\1102\classwork\A4-M2_실습자료
.env 존재 여부 : True
비밀번호 로드 : 성공
sales 문서 수 : 100


## 셀 2 · 결과를 보기 좋게 출력하는 도우미

In [9]:
import json

def show(res, n=5):
    """hits와 aggregations를 짧게 출력"""
    print("건수:", res["hits"]["total"]["value"])
    for h in res["hits"]["hits"][:n]:
        print("  ", h["_id"], h["_source"])
    if "aggregations" in res:
        print(json.dumps(res["aggregations"], ensure_ascii=False, indent=2))

## 셀 3 · 교안 10탭 — 지역별 매출 큰 순서

In [10]:
body = {
    "size": 0,
    "aggs": {"by_region": {
        "terms": {"field": "region", "order": {"rev": "desc"}},
        "aggs": {"rev": {"sum": {"field": "revenue"}}},
    }},
}
res = client.search(index="sales", body=body)
for b in res["aggregations"]["by_region"]["buckets"]:
    print(f'{b["key"]}  {b["doc_count"]:>2}건  {b["rev"]["value"]:>14,.0f}원')
# 서울 21건 58,566,750원 · 부산 16건 44,906,750원 · 광주 24건 39,029,500원 · 대구 23건 38,928,250원 · 대전 16건 31,861,500원

서울  21건      58,566,750원
부산  16건      44,906,750원
광주  24건      39,029,500원
대구  23건      38,928,250원
대전  16건      31,861,500원


## 셀 4 · Dev Tools 쿼리를 그대로 붙여 실행하는 연습
`03_연습문제_20.txt`의 문제를 Dev Tools에서 푼 뒤, 그 JSON을 아래 `body`에 붙여 같은 숫자가 나오는지 확인합니다.

In [ ]:
body = {
    # 여기에 Dev Tools에서 쓴 JSON 본문을 붙여넣기 (true/false/null → True/False/None)
    "size": 0,
    "query": {"range": {"order_date": {"gte": "2026-04-01", "lte": "2026-06-30"}}},
    "aggs": {"매출": {"sum": {"field": "revenue"}}},
}
show(client.search(index="sales", body=body))      # 연습 문제 5: 46건 / 89,782,500

## 셀 5 · 버킷을 DataFrame으로

In [ ]:
import pandas as pd

def buckets_to_df(res, agg_name, metric=None):
    rows = []
    for b in res["aggregations"][agg_name]["buckets"]:
        row = {"key": b.get("key_as_string", b["key"]), "doc_count": b["doc_count"]}
        if metric:
            row[metric] = b[metric]["value"]
        rows.append(row)
    return pd.DataFrame(rows)

res = client.search(index="sales", body={"size": 0, "aggs": {"월별": {
    "date_histogram": {"field": "order_date", "calendar_interval": "month", "format": "yyyy-MM"},
    "aggs": {"매출": {"sum": {"field": "revenue"}}}}}})
df = buckets_to_df(res, "월별", "매출")
print(df)
print("가장 큰 달:", df.loc[df["매출"].idxmax(), "key"])     # 연습 문제 15: 2026-03

## 셀 6 · 미션 5 — SQL을 Python에서 실행하고 DSL과 비교

In [ ]:
sql = "SELECT product, AVG(satisfaction) FROM sales WHERE region='대구' GROUP BY product"
r = client.transport.perform_request("POST", "/_plugins/_sql", body={"query": sql})
print("SQL:", r["datarows"])

body = {"size": 0, "query": {"term": {"region": "대구"}},
        "aggs": {"p": {"terms": {"field": "product"}, "aggs": {"sat": {"avg": {"field": "satisfaction"}}}}}}
res = client.search(index="sales", body=body)
print("DSL:", [(b["key"], round(b["sat"]["value"], 2)) for b in res["aggregations"]["p"]["buckets"]])

explain = client.transport.perform_request("POST", "/_plugins/_sql/_explain", body={"query": sql})
print(json.dumps(explain, ensure_ascii=False, indent=1)[:1500])

## 셀 7 · 미션 4 — JOIN 없이 두 인덱스 잇기 (Python)
`reviews` 인덱스가 있어야 합니다 (`01_reviews_인덱스_만들기_devtools.txt`).

In [ ]:
low = client.search(index="reviews", body={"query": {"range": {"rating": {"lte": 2}}},
                                            "_source": ["order_id", "rating", "comment"]})
ids = [h["_source"]["order_id"] for h in low["hits"]["hits"]]
print("평점 2점 이하 주문:", ids)                          # ['ORD022', 'ORD030']

orders = client.search(index="sales", body={"query": {"terms": {"order_id": ids}},
                                             "_source": ["order_id", "region", "product", "revenue", "status"]})
for h in orders["hits"]["hits"]:
    print(h["_source"])                                   # ORD022 배송중 / ORD030 취소